<a href="https://colab.research.google.com/github/Saibhossain/face-generation-model/blob/main/Text_to_Face_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers
!pip install transformers accelerate scipy safetensors


# explore the model (runwayml/stable-diffusion-v1-5)

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from huggingface_hub import model_info
import psutil
import os

# --- CONFIGURATION ---
MODEL_ID = "runwayml/stable-diffusion-v1-5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## function

In [ ]:
def print_section(title):
    print(f"\n{' '*20} {title.upper()} {' '*20}")

def get_hub_metadata():

    print_section("1. Hugging Face Hub Metadata")
    try:
        info = model_info(MODEL_ID)
        print(f"Model ID:       {info.modelId}")
        print(f"Author:         {info.author}")
        print(f"Downloads:      {info.downloads:,}")
        print(f"Likes:          {info.likes:,}")
        print(f"Library:        {info.library_name}")

        # Tags often contain license and task info
        print(f"Tags:           {', '.join(info.tags[:5])}...")

    except Exception as e:
        print(f"Error fetching Hub metadata: {e}")

def count_params(model):
    """Counts trainable parameters in a PyTorch model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def inspect_pipeline_components():
    """Loads the model and dissects its internals"""
    print_section("2. Architecture & Components")
    print(f"Loading model into {DEVICE} (float16 for efficiency)...")

    # Load pipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    # --- A. TEXT ENCODER (CLIP) ---

    print("This converts your prompt into vectors.")
    text_config = pipe.text_encoder.config
    print(f"   • Model Type:      {text_config.model_type}")
    print(f"   • Vocab Size:      {text_config.vocab_size:,}")
    print(f"   • Hidden Size:     {text_config.hidden_size}")
    print(f"   • Layers:          {text_config.num_hidden_layers}")
    print(f"   • Parameters:      {count_params(pipe.text_encoder):,}")

    # --- B. UNET (The Noise Predictor) ---
    print("\n[Component 2: UNet (The Engine)] ")
    print("This predicts noise to subtract from the image.")
    unet_config = pipe.unet.config
    print(f"   • Sample Size:     {unet_config.sample_size} (Internal processing resolution)")
    print(f"   • In Channels:     {unet_config.in_channels} (Latent channels)")
    print(f"   • Cross Attn Dim:  {unet_config.cross_attention_dim} (Text embedding size)")
    print(f"   • Attn Head Dim:   {unet_config.attention_head_dim}")
    print(f"   • Parameters:      {count_params(pipe.unet):,}")

    # --- C. VAE (Variational Autoencoder) ---
    print("\n[Component 3: VAE (The Compressor)] ")
    print("This compresses images to latents and decompresses them back.")
    vae_config = pipe.vae.config
    print(f"   • Latent Channels: {vae_config.latent_channels}")
    print(f"   • Block Out Ch:    {vae_config.block_out_channels}")
    print(f"   • Parameters:      {count_params(pipe.vae):,}")

    # --- D. SCHEDULER ---
    print("\n[Component 4: Scheduler]")
    print(f"   • Default Type:    {pipe.scheduler.__class__.__name__}")
    print(f"   • Timesteps:       {pipe.scheduler.config.num_train_timesteps} (Training steps)")
    print(f"   • Beta Schedule:   {pipe.scheduler.config.beta_schedule}")

    return pipe

def qualitative_test(pipe):
    """Runs a generation test"""
    print_section("3. Qualitative Evaluation (Inference Test)")

    prompt = "A high-tech robot painting a canvas, detailed, 8k, cyberpunk style"
    print(f"Generating image for prompt: '{prompt}'")

    image = pipe(prompt, num_inference_steps=25).images[0]

    save_path = "sd15_evaluation_sample.png"
    image.save(save_path)
    print(f"Test image saved to: {save_path}")
    print("Check this image to evaluate visual quality.")

## METADATA

In [ ]:
get_hub_metadata()

## ARCHITECTURE

In [ ]:
pipeline_obj = inspect_pipeline_components()

## TEST

In [ ]:
qualitative_test(pipeline_obj)

# my code

In [ ]:
import torch
from IPython.display import display
import sys
import subprocess
from diffusers import StableDiffusionPipeline

# --- 2. LOAD MODEL ---
# We use Stable Diffusion v1.5 because it is excellent at portraits and human faces
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Model: {model_id}...")
# float16 makes it faster and use less memory on Colab GPUs
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to(device)

# Optimization for faster generation
pipe.enable_attention_slicing()

# --- 3. GENERATION FUNCTION ---
def generate_face(prompt, negative_prompt="cartoon, 3d, disfigured, bad art"):
    print(f"\n✨ Generating: '{prompt}'...")

    # The pipeline handles all the math (denoising, latents, decoding) automatically
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        height=512,
        width=512,
        num_inference_steps=30 # 30 is a sweet spot for speed/quality
    ).images[0]

    display(image)
    return image

# --- 4. USER INPUT ---
# Type your description here!
user_prompt = "A cinematic portrait of a futuristic cyberpunk warrior with neon blue eyes, 8k resolution, photorealistic"


# Run Generation
img = generate_face(user_prompt)